In [0]:
%sql
--On-time rate by store and hour
SELECT store_id, placed_hour,
       COUNT(*) AS delivered_orders,
       ROUND(100 * AVG(CASE WHEN is_on_time THEN 1.0 ELSE 0 END), 1) AS on_time_pct
FROM workspace.gold.fact_orders
WHERE order_status = 'delivered'
GROUP BY store_id, placed_hour
ORDER BY store_id, placed_hour;

In [0]:
%sql
--Which stage causes the delay at dinner peak? (average minutes per stage)
SELECT CASE WHEN placed_hour IN (19, 20, 21) THEN 'dinner 19-21h' ELSE 'other hours' END AS period,
       ROUND(AVG(accept_min), 2)     AS accept,
       ROUND(AVG(pick_wait_min), 2)  AS pick_wait,
       ROUND(AVG(pick_pack_min), 2)  AS pick_and_pack,
       ROUND(AVG(rider_wait_min), 2) AS rider_wait,
       ROUND(AVG(ride_min), 2)       AS ride,
       ROUND(AVG(delivery_minutes), 2) AS total
FROM workspace.gold.fact_orders
WHERE order_status = 'delivered' AND NOT is_split
GROUP BY 1
ORDER BY 1;

In [0]:
%sql
--Longest rider waits: store x hour (top 10)
SELECT store_id, placed_hour,
       COUNT(*) AS orders,
       ROUND(AVG(rider_wait_min), 2) AS avg_rider_wait_min
FROM workspace.gold.fact_orders
WHERE order_status = 'delivered'
GROUP BY store_id, placed_hour
ORDER BY avg_rider_wait_min DESC
LIMIT 10;

In [0]:
%sql
--Fill rate by store and category, strict vs lenient (10 lowest)
SELECT o.store_id, s.category,
       ROUND(100.0 * SUM(CASE WHEN i.item_status = 'fulfilled' THEN i.qty_ordered ELSE 0 END)
             / SUM(i.qty_ordered), 2) AS strict_fill_pct,
       ROUND(100.0 * SUM(CASE WHEN i.item_status IN ('fulfilled', 'substituted', 'fulfilled_split')
                              THEN i.qty_ordered ELSE 0 END)
             / SUM(i.qty_ordered), 2) AS lenient_fill_pct
FROM workspace.silver.order_items i
JOIN workspace.silver.orders o ON i.order_id = o.order_id
JOIN workspace.bronze.skus s   ON i.sku_id = s.sku_id
WHERE o.order_status = 'delivered'
GROUP BY o.store_id, s.category
ORDER BY strict_fill_pct
LIMIT 10;

In [0]:
%sql
--SKUs with the most unavailable lines and removed value (top 10)
SELECT a.sku_id, a.category,
       SUM(a.unavailable_lines) AS unavailable_lines,
       SUM(a.removed_value)     AS removed_value
FROM workspace.gold.sku_availability a
GROUP BY a.sku_id, a.category
ORDER BY removed_value DESC
LIMIT 10;

In [0]:
%sql
--Cancellations by reason and store
SELECT store_id, cancel_reason, COUNT(*) AS cancelled_orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY store_id), 1) AS pct_of_store_cancellations
FROM workspace.gold.fact_orders
WHERE order_status = 'cancelled'
GROUP BY store_id, cancel_reason
ORDER BY store_id, cancelled_orders DESC;

In [0]:
%sql
--Fruit and veg: wastage vs stockouts by store
SELECT store_id,
       ROUND(100.0 * SUM(qty_wasted) / SUM(qty_received), 1) AS wastage_pct,
       ROUND(100.0 * SUM(oos_sku_hours) / SUM(sku_hours), 1) AS stockout_pct
FROM workspace.gold.fv_store_day
GROUP BY store_id
ORDER BY wastage_pct DESC;

In [0]:
%sql
--Do substitutions reduce cancellations? (non-split orders with a missing item)
WITH ev AS (
  SELECT order_id,
         SUM(CASE WHEN event_type = 'item_unavailable' THEN 1 ELSE 0 END) AS n_unavailable,
         SUM(CASE WHEN event_type = 'item_substituted' THEN 1 ELSE 0 END) AS n_substituted
  FROM workspace.silver.order_events
  GROUP BY order_id
)
SELECT CASE WHEN ev.n_substituted = ev.n_unavailable THEN 'all missing items replaced'
            ELSE 'at least one missing item not replaced' END AS outcome,
       COUNT(*) AS orders,
       ROUND(100.0 * AVG(CASE WHEN f.order_status = 'cancelled' THEN 1.0 ELSE 0 END), 1) AS cancel_pct
FROM ev
JOIN workspace.gold.fact_orders f ON ev.order_id = f.order_id
WHERE ev.n_unavailable > 0 AND NOT f.is_split
GROUP BY 1;

In [0]:
%sql
--Reorder within 7 days after a good vs bad first order
SELECT first_order_bad,
       COUNT(*) AS customers,
       ROUND(100.0 * AVG(CASE WHEN reordered_7d THEN 1.0 ELSE 0 END), 1) AS reorder_7d_pct
FROM workspace.gold.customer_first_order
WHERE has_full_7d_window
GROUP BY first_order_bad;

In [0]:
%sql
--Rain vs dry
SELECT is_rainy,
       COUNT(*) AS orders,
       ROUND(100.0 * AVG(CASE WHEN order_status = 'cancelled' THEN 1.0 ELSE 0 END), 2) AS cancel_pct,
       ROUND(100.0 * AVG(CASE WHEN is_on_time THEN 1.0 WHEN is_on_time = FALSE THEN 0.0 END), 1) AS on_time_pct,
       ROUND(AVG(ride_min), 2) AS avg_ride_min
FROM workspace.gold.fact_orders
GROUP BY is_rainy;

In [0]:
%sql
--Split vs normal orders
SELECT is_split,
       COUNT(*) AS orders,
       ROUND(AVG(delivery_minutes), 2) AS avg_delivery_minutes,
       ROUND(100.0 * AVG(CASE WHEN order_status = 'cancelled' THEN 1.0 ELSE 0 END), 1) AS cancel_pct,
       ROUND(100.0 * AVG(CASE WHEN is_on_time THEN 1.0 WHEN is_on_time = FALSE THEN 0.0 END), 1) AS on_time_pct
FROM workspace.gold.fact_orders
GROUP BY is_split;

In [0]:
%sql
--Do a few SKUs cause most of the removed value?
WITH sku_loss AS (
  SELECT sku_id, SUM(removed_value) AS lost
  FROM workspace.gold.sku_availability
  GROUP BY sku_id
  HAVING SUM(removed_value) > 0
),
ranked AS (
  SELECT sku_id, lost,
         ROW_NUMBER() OVER (ORDER BY lost DESC) AS rk,
         COUNT(*) OVER ()                       AS n_skus,
         SUM(lost) OVER (ORDER BY lost DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
           / SUM(lost) OVER ()                  AS cum_share
  FROM sku_loss
)
SELECT MIN(rk) AS skus_needed_for_80pct, MAX(n_skus) AS skus_with_loss,
       ROUND(100.0 * MIN(rk) / MAX(n_skus), 1) AS pct_of_skus
FROM ranked
WHERE cum_share >= 0.8;